# Tokenizer Pipeline (Sharded) — synapse

This notebook is a **thin Colab wrapper** around the standalone scripts in `tokenize/` of the [synapse repo](https://github.com/ajencinas/synapse). The actual logic lives in `run_tokenizer.py` (training + sharding + auto-merge) and `merge_shards.py` (order-preserving shard merger). The notebook only:

1. Mounts Drive
2. Pulls the latest `run_tokenizer.py`, `merge_shards.py`, and `tokenizer_config.json` from GitHub
3. Shells out to `!python run_tokenizer.py` with env vars pointing at Drive

Pipeline (executed by `run_tokenizer.py`):

1. **Load existing tokenizer** from `tokenizer_out/tokenizer.json` (default), or train a new Byte-Level BPE (vocab 64k, digit-per-token, 256 specials) when `--train` is passed
2. **Auto-eval gate**: encode a held-out sample and abort if compression ratios look broken — protects you from sharding 42B tokens with a bad tokenizer
3. Tokenize each source `.txt` into uint16 shards in `token_shards/` (only new/changed files are re-tokenized)
4. **Auto-merge**: stream-concatenate per-source shards within each domain into ~100 MB uniform shards in `token_shards_merged/` (preserves source order)

Edit `tokenizer_config.json` (in the repo) to change vocab size, training budget, eval thresholds, or merge target size. Push, then re-run cell 2.

Pass `--train`, `--no-merge`, `--workers N`, or `--merge-target-bytes N` via `EXTRA_ARGS` in cell 3 to tune the run. **First-time runs (no existing `tokenizer.json`) must use `--train`.**

In [ ]:
# ---- 1. Deps + mount Drive ----
!pip install -q tokenizers numpy

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- 2. Pull pipeline scripts + config from GitHub ----
# Override BRANCH to pin a specific commit, branch, or tag.
import os, urllib.request

REPO   = 'ajencinas/synapse'
BRANCH = 'main'
FILES  = [
    'tokenize/run_tokenizer.py',
    'tokenize/merge_shards.py',
    'tokenize/tokenizer_config.json',
]

os.chdir('/content')
for path in FILES:
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{path}'
    dst = os.path.basename(path)
    print(f'  fetch {url}')
    urllib.request.urlretrieve(url, dst)
    print(f'    -> /content/{dst} ({os.path.getsize(dst):,} bytes)')

print('\nUsing config:')
!head -40 tokenizer_config.json

In [ ]:
# ---- 3. Run the pipeline (load tokenizer -> eval gate -> shard -> auto-merge) ----
# Drive layout: /content/drive/MyDrive/synapse/{datasets_pretrain,token_shards,...}
# After Step 4 (auto-merge), uniform 100 MB shards land in token_shards_merged/.
import os

SYNAPSE = '/content/drive/MyDrive/synapse'
os.environ['TOKENIZER_DATA_PATH']    = f'{SYNAPSE}/datasets_pretrain'
os.environ['TOKENIZER_OUT_DIR']      = f'{SYNAPSE}/tokenizer_out'
os.environ['TOKENIZER_SHARD_DIR']    = f'{SYNAPSE}/token_shards'
os.environ['TOKENIZER_MANIFEST_DIR'] = f'{SYNAPSE}/manifests'
os.environ['TOKENIZER_CONFIG_PATH']  = '/content/tokenizer_config.json'

# Default: load existing tokenizer + shard new/changed files + merge.
# Override here to tweak the run. Examples:
#   EXTRA_ARGS = '--train'                             # train a new tokenizer (REQUIRED on first run)
#   EXTRA_ARGS = '--no-merge'                          # skip auto-merge step
#   EXTRA_ARGS = '--workers 4 --merge-target-bytes 209715200'   # 200 MB shards, 4 workers
EXTRA_ARGS = ''

!python /content/run_tokenizer.py {EXTRA_ARGS}